In [ ]:
using Pkg
Pkg.activate("..")
using Revise

In [ ]:
using bslLD, CairoMakie, Statistics
using FFTW, DSP
bslLD.greet()

bslLD.use_cuda!()

In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[128,65,65],0.02,10000,1)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)

initFuncx(x) = 1+0.00001*randn()
f = bslLD.Distribution(grid, 0.00001,initFuncv=initFuncv, initFuncx=initFuncx);
rho = bslLD.compute_density(f,grid)

e = bslLD.empty_vectorfield(grid);


In [ ]:
df = (x-> x.-mean(x,dims = 1))(Array(f.data))
heatmap(df[:,:,33])

In [ ]:
function step(f,grid)
    bslLD.advectX!(f,grid)
    ex = @. 0.05 * sin(2pi * grid.xaxes[1] / grid.max[1])
    e = bslLD.VectorField([collect(ex)])
    bslLD.advectV!(f,grid,e)
end

function stepSelfConsitent(f,grid)
    bslLD.advectX!(f,grid)
    rho = bslLD.compute_density(f,grid)
    phi = -1*bslLD.adiabatic(rho,grid)
    e = bslLD.compute_e(phi,grid)
    bslLD.advectV!(f,grid,e)
end

In [ ]:
step(f,grid)

In [ ]:
rhodiag = []
fdiag = []


for i in grid.itime
    grid.index[1] = i
    stepSelfConsitent(f,grid)
    if i%1==0
        push!(fdiag,f.data)
        push!(rhodiag, bslLD.compute_density(f,grid).data)
    end
end


In [ ]:
locData = hcat(map(x-> x.-mean(x), Array.(rhodiag))...)


Nx, Ny = size(locData)
w = kaiser(Ny, 3)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

heatmap(log.(abs.(fft(windowed)[1:round(Int,Nx/2), 1:100])))